# Sports RAG - Retriever Phase Experiments

This notebook implements the retrieval phase exactly for your plan:
- BM25 on fixed chunks
- BM25 on recursive chunks
- BM25 on semantic chunks
- Hybrid (BM25 fixed + dense fixed namespace) with RRF
- Hybrid (BM25 recursive + dense recursive namespace) with RRF
- Hybrid (BM25 semantic + dense semantic namespace) with RRF

Then it evaluates all retrieval setups on 25 manual questions with Recall@5, Recall@10, and MRR@10, selects winners, and runs QA comparison on only those winners.

## 1) Install Dependencies

In [1]:
!pip -q install rank-bm25 sentence-transformers pinecone pandas numpy tqdm transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 742.7/742.7 kB 13.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.7/280.7 kB 24.5 MB/s eta 0:00:00


## 2) Imports and Global Config

In [2]:
import os
import json
import re
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from tqdm import tqdm
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone
from transformers import pipeline

# Retrieval hyperparameters
TOP_K = 10
RRF_K = 60
DENSE_TOP_K = 50

# Embedding model for dense retrieval queries (good quality + Kaggle-friendly size)
EMBED_MODEL_NAME = "BAAI/bge-small-en-v1.5"

# QA model for final winner-vs-winner answer comparison
QA_MODEL_NAME = "google/flan-t5-base"

# Kaggle Secrets support for Pinecone API key.
# If running on Kaggle and secret exists, expose it via env var expected by the notebook.
try:
    from kaggle_secrets import UserSecretsClient

    user_secrets = UserSecretsClient()
    secret_value_0 = user_secrets.get_secret("PINECONE_API_KEY")
    if secret_value_0:
        os.environ["PINECONE_API_KEY"] = secret_value_0
        print("Loaded PINECONE_API_KEY from Kaggle Secrets.")
except Exception:
    # Local environment or missing Kaggle Secrets; env var can still be set manually.
    pass

print("Imports and config loaded.")

Loaded PINECONE_API_KEY from Kaggle Secrets.
Imports and config loaded.


## 3) Locate Input Files and Load Chunks

In [3]:
def resolve_chunk_dir() -> Path:
    cwd = Path.cwd()

    # Preferred local/Kaggle working directory names
    candidate_dirs = [
        cwd / "chunks_BM25",
        cwd / "Keyword Chunks",
        cwd.parent / "chunks_BM25",
        cwd.parent / "Keyword Chunks",
    ]

    for d in candidate_dirs:
        if d.exists():
            return d

    # Kaggle dataset mount fallback: /kaggle/input/<dataset>/...
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        # First, look for a directory named chunks_BM25 anywhere under /kaggle/input
        for d in kaggle_input.rglob("chunks_BM25"):
            if d.is_dir():
                return d

        # Then, look for any directory that contains all three required files
        required = {"chunks_fixed.json", "chunks_recursive.json", "chunks_semantic.json"}
        for fixed_file in kaggle_input.rglob("chunks_fixed.json"):
            parent = fixed_file.parent
            names = {p.name for p in parent.glob("*.json")}
            if required.issubset(names):
                return parent

    raise FileNotFoundError(
        "Could not find chunk JSON folder. Expected one of: "
        "./chunks_BM25, ./Keyword Chunks, ../chunks_BM25, ../Keyword Chunks, "
        "or a Kaggle input directory containing chunks_fixed.json/chunks_recursive.json/chunks_semantic.json."
    )

CHUNK_DIR = resolve_chunk_dir()
PROJECT_ROOT = CHUNK_DIR.parent

chunk_paths = {
    "fixed": CHUNK_DIR / "chunks_fixed.json",
    "recursive": CHUNK_DIR / "chunks_recursive.json",
    "semantic": CHUNK_DIR / "chunks_semantic.json",
}

chunks_by_strategy = {}
for strategy, path in chunk_paths.items():
    with open(path, "r", encoding="utf-8") as f:
        chunks_by_strategy[strategy] = json.load(f)

print(f"Chunk directory: {CHUNK_DIR}")
for strategy, chunks in chunks_by_strategy.items():
    print(f"{strategy}: {len(chunks)} chunks")

# Build id->chunk lookup per strategy
chunk_lookup = {
    strategy: {item["id"]: item for item in items}
    for strategy, items in chunks_by_strategy.items()
}

Chunk directory: /kaggle/input/datasets/muhammadhamzaarif/chunks-bm25
fixed: 4872 chunks
recursive: 4860 chunks
semantic: 4998 chunks


## 4) Build BM25 Indexes (Fixed, Recursive, Semantic)

In [4]:
def bm25_tokenize(text: str):
    return re.findall(r"[A-Za-z0-9]+", text.lower())

bm25_indexes = {}
for strategy, chunks in chunks_by_strategy.items():
    tokenized_corpus = [bm25_tokenize(c["text"]) for c in chunks]
    bm25_indexes[strategy] = BM25Okapi(tokenized_corpus)

print("Built BM25 indexes for fixed/recursive/semantic.")

Built BM25 indexes for fixed/recursive/semantic.


## 5) BM25 and Dense Retrieval Helpers

In [5]:
def bm25_retrieve(query: str, strategy: str, top_k: int = TOP_K):
    chunks = chunks_by_strategy[strategy]
    bm25 = bm25_indexes[strategy]

    q_tokens = bm25_tokenize(query)
    scores = bm25.get_scores(q_tokens)
    ranked_idx = np.argsort(scores)[::-1][:top_k]

    results = []
    for idx in ranked_idx:
        chunk = chunks[idx]
        results.append({
            "id": chunk["id"],
            "score": float(scores[idx]),
            "text": chunk.get("text", ""),
            "source": chunk.get("source", ""),
            "url": chunk.get("url", ""),
            "strategy": strategy,
        })
    return results

def init_dense_components(index_name: str = "sports-rag"):
    pinecone_api_key = os.environ.get("PINECONE_API_KEY")
    if not pinecone_api_key:
        raise EnvironmentError("Set PINECONE_API_KEY before running dense retrieval.")

    pc = Pinecone(api_key=pinecone_api_key)
    index = pc.Index(index_name)
    embed_model = SentenceTransformer(EMBED_MODEL_NAME)
    return index, embed_model

def dense_retrieve(query: str, namespace: str, strategy: str, index, embed_model, top_k: int = TOP_K):
    q_vec = embed_model.encode(query, normalize_embeddings=True).tolist()

    response = index.query(
        vector=q_vec,
        top_k=top_k,
        namespace=namespace,
        include_metadata=True,
    )

    local_lookup = chunk_lookup[strategy]
    results = []

    for match in response.get("matches", []):
        match_id = match.get("id")
        metadata = match.get("metadata", {}) or {}
        fallback = local_lookup.get(match_id, {})

        results.append({
            "id": match_id,
            "score": float(match.get("score", 0.0)),
            "text": metadata.get("text", fallback.get("text", "")),
            "source": metadata.get("source", fallback.get("source", "")),
            "url": metadata.get("url", fallback.get("url", "")),
            "strategy": strategy,
        })
    return results

## 6) Reciprocal Rank Fusion (RRF) and Hybrid Retriever

In [6]:
def reciprocal_rank_fusion(result_lists, rrf_k: int = RRF_K, top_k: int = TOP_K):
    fused_scores = defaultdict(float)
    payload = {}

    for results in result_lists:
        for rank, item in enumerate(results, start=1):
            doc_id = item["id"]
            fused_scores[doc_id] += 1.0 / (rrf_k + rank)
            if doc_id not in payload:
                payload[doc_id] = item

    ranked = sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]

    fused = []
    for doc_id, score in ranked:
        item = payload[doc_id].copy()
        item["score"] = float(score)
        fused.append(item)
    return fused

def hybrid_retrieve(query: str, strategy: str, namespace: str, index, embed_model, top_k: int = TOP_K):
    sparse_hits = bm25_retrieve(query, strategy=strategy, top_k=DENSE_TOP_K)
    dense_hits = dense_retrieve(
        query,
        namespace=namespace,
        strategy=strategy,
        index=index,
        embed_model=embed_model,
        top_k=DENSE_TOP_K,
    )
    return reciprocal_rank_fusion([sparse_hits, dense_hits], rrf_k=RRF_K, top_k=top_k)

## 7) Sanity Check Queries

In [7]:
sample_query = "Who won the 2022 FIFA World Cup final?"

print("BM25 fixed sample:")
for r in bm25_retrieve(sample_query, strategy="fixed", top_k=3):
    print(r["id"], "|", r["score"])

BM25 fixed sample:
fixed_Argentina_national_football_team_1 | 15.051653894026892
fixed_Kylian_Mbappe_3 | 14.83107891177794
fixed_Manchester_City_FC_20 | 14.566597227609407


## 8) Define 25 Evaluation Questions

Fill this list with exactly 25 manually created questions.
Each item must contain:
- question: string
- relevant_chunk_ids: list of acceptable chunk IDs for relevance (from the chosen strategy files)

In [8]:
evaluation_questions = [
    {
        "question": "What is the FIFA World Cup?",
        "relevant_chunk_ids": [
            "fixed_FIFA_World_Cup_0",
            "recursive_FIFA_World_Cup_0",
            "semantic_FIFA_World_Cup_0",
        ],
    },
    {
        "question": "Which tournament is the 2022 FIFA World Cup?",
        "relevant_chunk_ids": [
            "fixed_2022_FIFA_World_Cup_0",
            "recursive_2022_FIFA_World_Cup_0",
            "semantic_2022_FIFA_World_Cup_0",
        ],
    },
    {
        "question": "Which tournament is the 2018 FIFA World Cup?",
        "relevant_chunk_ids": [
            "fixed_2018_FIFA_World_Cup_0",
            "recursive_2018_FIFA_World_Cup_0",
            "semantic_2018_FIFA_World_Cup_0",
        ],
    },
    {
        "question": "What is FIFA in world football?",
        "relevant_chunk_ids": [
            "fixed_FIFA_0",
            "recursive_FIFA_0",
            "semantic_FIFA_0",
        ],
    },
    {
        "question": "What club is AC Milan?",
        "relevant_chunk_ids": [
            "fixed_AC_Milan_0",
            "recursive_AC_Milan_0",
            "semantic_AC_Milan_0",
        ],
    },
    {
        "question": "What club is AFC Ajax?",
        "relevant_chunk_ids": [
            "fixed_AFC_Ajax_0",
            "recursive_AFC_Ajax_0",
            "semantic_AFC_Ajax_0",
        ],
    },
    {
        "question": "What competition is the AFC Champions League Elite?",
        "relevant_chunk_ids": [
            "fixed_AFC_Champions_League_Elite_0",
            "recursive_AFC_Champions_League_Elite_0",
            "semantic_AFC_Champions_League_Elite_0",
        ],
    },
    {
        "question": "Which country does the Argentina national football team represent?",
        "relevant_chunk_ids": [
            "fixed_Argentina_national_football_team_0",
            "recursive_Argentina_national_football_team_0",
            "semantic_Argentina_national_football_team_0",
        ],
    },
    {
        "question": "What club is Arsenal F.C.?",
        "relevant_chunk_ids": [
            "fixed_Arsenal_FC_0",
            "recursive_Arsenal_FC_0",
            "semantic_Arsenal_FC_0",
        ],
    },
    {
        "question": "What is the Ballon d'Or award in football?",
        "relevant_chunk_ids": [
            "fixed_Ballon_dOr_0",
            "recursive_Ballon_dOr_0",
            "semantic_Ballon_dOr_0",
        ],
    },
    {
        "question": "What club is Borussia Dortmund?",
        "relevant_chunk_ids": [
            "fixed_Borussia_Dortmund_0",
            "recursive_Borussia_Dortmund_0",
            "semantic_Borussia_Dortmund_0",
        ],
    },
    {
        "question": "Which country does the Brazil national football team represent?",
        "relevant_chunk_ids": [
            "fixed_Brazil_national_football_team_0",
            "recursive_Brazil_national_football_team_0",
            "semantic_Brazil_national_football_team_0",
        ],
    },
    {
        "question": "What is the Bundesliga?",
        "relevant_chunk_ids": [
            "fixed_Bundesliga_0",
            "recursive_Bundesliga_0",
            "semantic_Bundesliga_0",
        ],
    },
    {
        "question": "What club is Chelsea F.C.?",
        "relevant_chunk_ids": [
            "fixed_Chelsea_FC_0",
            "recursive_Chelsea_FC_0",
            "semantic_Chelsea_FC_0",
        ],
    },
    {
        "question": "What competition is Copa Libertadores?",
        "relevant_chunk_ids": [
            "fixed_Copa_Libertadores_0",
            "recursive_Copa_Libertadores_0",
            "semantic_Copa_Libertadores_0",
        ],
    },
    {
        "question": "Who is Cristiano Ronaldo?",
        "relevant_chunk_ids": [
            "fixed_Cristiano_Ronaldo_0",
            "recursive_Cristiano_Ronaldo_0",
            "semantic_Cristiano_Ronaldo_0",
        ],
    },
    {
        "question": "Which country does the Croatia national football team represent?",
        "relevant_chunk_ids": [
            "fixed_Croatia_national_football_team_0",
            "recursive_Croatia_national_football_team_0",
            "semantic_Croatia_national_football_team_0",
        ],
    },
    {
        "question": "Who is Diego Maradona?",
        "relevant_chunk_ids": [
            "fixed_Diego_Maradona_0",
            "recursive_Diego_Maradona_0",
            "semantic_Diego_Maradona_0",
        ],
    },
    {
        "question": "What is the EFL Cup?",
        "relevant_chunk_ids": [
            "fixed_EFL_Cup_0",
            "recursive_EFL_Cup_0",
            "semantic_EFL_Cup_0",
        ],
    },
    {
        "question": "Which country does the England national football team represent?",
        "relevant_chunk_ids": [
            "fixed_England_national_football_team_0",
            "recursive_England_national_football_team_0",
            "semantic_England_national_football_team_0",
        ],
    },
    {
        "question": "What is the Eredivisie?",
        "relevant_chunk_ids": [
            "fixed_Eredivisie_0",
            "recursive_Eredivisie_0",
            "semantic_Eredivisie_0",
        ],
    },
    {
        "question": "Who is Erling Haaland?",
        "relevant_chunk_ids": [
            "fixed_Erling_Haaland_0",
            "recursive_Erling_Haaland_0",
            "semantic_Erling_Haaland_0",
        ],
    },
    {
        "question": "What is the FA Cup?",
        "relevant_chunk_ids": [
            "fixed_FA_Cup_0",
            "recursive_FA_Cup_0",
            "semantic_FA_Cup_0",
        ],
    },
    {
        "question": "What club is FC Barcelona?",
        "relevant_chunk_ids": [
            "fixed_FC_Barcelona_0",
            "recursive_FC_Barcelona_0",
            "semantic_FC_Barcelona_0",
        ],
    },
    {
        "question": "What club is FC Bayern Munich?",
        "relevant_chunk_ids": [
            "fixed_FC_Bayern_Munich_0",
            "recursive_FC_Bayern_Munich_0",
            "semantic_FC_Bayern_Munich_0",
        ],
    },
]

print("Current #questions:", len(evaluation_questions))
if len(evaluation_questions) != 25:
    print("WARNING: Add questions until you have exactly 25 for final evaluation.")
else:
    print("All 25 football-only evaluation questions loaded with real chunk IDs.")

Current #questions: 25
All 25 football-only evaluation questions loaded with real chunk IDs.


## 9) Evaluation Metrics: Recall@5, Recall@10, MRR@10

In [9]:
def compute_recall_at_k(pred_ids, relevant_ids, k):
    top_pred = pred_ids[:k]
    hits = len(set(top_pred).intersection(set(relevant_ids)))
    return 1.0 if hits > 0 else 0.0

def compute_mrr_at_k(pred_ids, relevant_ids, k=10):
    relevant = set(relevant_ids)
    for rank, doc_id in enumerate(pred_ids[:k], start=1):
        if doc_id in relevant:
            return 1.0 / rank
    return 0.0

def evaluate_retriever(eval_data, retriever_fn, top_k=10):
    rows = []
    for item in tqdm(eval_data, desc="Evaluating"):
        q = item["question"]
        rel = item["relevant_chunk_ids"]

        results = retriever_fn(q, top_k)
        pred_ids = [r["id"] for r in results]

        rows.append({
            "question": q,
            "recall@5": compute_recall_at_k(pred_ids, rel, 5),
            "recall@10": compute_recall_at_k(pred_ids, rel, 10),
            "mrr@10": compute_mrr_at_k(pred_ids, rel, 10),
        })

    df = pd.DataFrame(rows)
    return {
        "Recall@5": df["recall@5"].mean(),
        "Recall@10": df["recall@10"].mean(),
        "MRR@10": df["mrr@10"].mean(),
        "details": df,
    }

## 10) Run All 6 Retrieval Experiments

In [10]:
experiments = {
    "bm25_fixed": lambda q, k: bm25_retrieve(q, "fixed", top_k=k),
    "bm25_recursive": lambda q, k: bm25_retrieve(q, "recursive", top_k=k),
    "bm25_semantic": lambda q, k: bm25_retrieve(q, "semantic", top_k=k),
}

dense_ready = bool(os.environ.get("PINECONE_API_KEY"))
if dense_ready:
    index, embed_model = init_dense_components(index_name="sports-rag")
    experiments.update({
        "hybrid_fixed": lambda q, k: hybrid_retrieve(q, "fixed", "fixed", index, embed_model, top_k=k),
        "hybrid_recursive": lambda q, k: hybrid_retrieve(q, "recursive", "recursive", index, embed_model, top_k=k),
        "hybrid_semantic": lambda q, k: hybrid_retrieve(q, "semantic", "semantic", index, embed_model, top_k=k),
    })
else:
    print("PINECONE_API_KEY not set: running BM25-only experiments.")

all_results = []
for name, retriever_fn in experiments.items():
    metrics = evaluate_retriever(evaluation_questions, retriever_fn, top_k=10)
    all_results.append({
        "experiment": name,
        "Recall@5": metrics["Recall@5"],
        "Recall@10": metrics["Recall@10"],
        "MRR@10": metrics["MRR@10"],
    })

results_df = pd.DataFrame(all_results).sort_values("MRR@10", ascending=False)
results_df

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Evaluating: 100%|██████████| 25/25 [00:05<00:00,  4.46it/s]


,experiment,Recall@5,Recall@10,MRR@10
5,hybrid_semantic,0.80,0.92,0.716111
3,hybrid_fixed,0.80,0.88,0.713000
4,hybrid_recursive,0.76,0.88,0.711667
2,bm25_semantic,0.60,0.68,0.311556
0,bm25_fixed,0.48,0.60,0.281714
1,bm25_recursive,0.48,0.64,0.281048


## 11) Winner Selection (1 Best Sparse + 1 Best Hybrid)

In [11]:
sparse_df = results_df[results_df["experiment"].str.startswith("bm25_")].copy()
hybrid_df = results_df[results_df["experiment"].str.startswith("hybrid_")].copy()

best_sparse = sparse_df.iloc[0]["experiment"] if not sparse_df.empty else None
best_hybrid = hybrid_df.iloc[0]["experiment"] if not hybrid_df.empty else None

print("Best sparse:", best_sparse)
print("Best hybrid:", best_hybrid)

Best sparse: bm25_semantic
Best hybrid: hybrid_semantic


## 12) Final QA Phase on Winners Only

In [12]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

def build_context(results, max_chars=2500):
    context_parts = []
    total = 0
    for r in results:
        txt = r.get("text", "")
        if not txt:
            continue
        if total + len(txt) > max_chars:
            break
        context_parts.append(txt)
        total += len(txt)
    return "\n\n".join(context_parts)

def make_qa_prompt(question, context):
    return (
        "Answer the question using only the context. If context is insufficient, say: I don't know.\n\n"
        f"Question: {question}\n\n"
        f"Context:\n{context}\n\n"
        "Answer:"
    )

# Robust QA loading for newer Transformers versions where text2text pipeline task may be unavailable
qa_tokenizer = AutoTokenizer.from_pretrained(QA_MODEL_NAME)
qa_model = AutoModelForSeq2SeqLM.from_pretrained(QA_MODEL_NAME)
qa_device = "cuda" if torch.cuda.is_available() else "cpu"
qa_model = qa_model.to(qa_device)

def answer_with_retriever(question, retriever_fn):
    hits = retriever_fn(question, 5)
    context = build_context(hits)
    prompt = make_qa_prompt(question, context)

    inputs = qa_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024,
    )
    inputs = {k: v.to(qa_device) for k, v in inputs.items()}

    with torch.no_grad():
        output_ids = qa_model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
        )

    out = qa_tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()
    return out

winner_map = {}
for name in [best_sparse, best_hybrid]:
    if name is not None and name in experiments:
        winner_map[name] = experiments[name]

qa_rows = []
for item in tqdm(evaluation_questions, desc="QA Comparison"):
    q = item["question"]
    row = {"question": q}
    for winner_name, retriever_fn in winner_map.items():
        row[f"answer_{winner_name}"] = answer_with_retriever(q, retriever_fn)
    qa_rows.append(row)

qa_df = pd.DataFrame(qa_rows)
qa_df.head()

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

QA Comparison: 100%|██████████| 25/25 [00:14<00:00,  1.70it/s]


,question,answer_bm25_semantic,answer_hybrid_semantic
0,What is the FIFA World Cup?,sport,international association football competition
1,Which tournament is the 2022 FIFA World Cup?,World Cup,World Cup
2,Which tournament is the 2018 FIFA World Cup?,World Cup tournament,2022
3,What is FIFA in world football?,FIFA,governing body
4,What club is AC Milan?,Inter Milan,Milan Foot-Ball and Cricket Club


## 13) Save Outputs

In [13]:
output_dir = PROJECT_ROOT / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

if 'results_df' in globals():
    results_df.to_csv(output_dir / "retrieval_metrics.csv", index=False)

if 'qa_df' in globals():
    qa_df.to_csv(output_dir / "qa_winner_comparison.csv", index=False)

print(f"Saved outputs in: {output_dir}")

OSError: [Errno 30] Read-only file system: '/kaggle/input/datasets/muhammadhamzaarif/outputs'